In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.


In [1]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline











   ---------------------------------------- 0.0/42.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/42.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/42.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/42.9 MB 776.4 kB/s eta 0:00:55
    --------------------------------------- 0.8/42.9 MB 970.8 kB/s eta 0:00:44
    --------------------------------------- 1.0/42.9 MB 1.1 MB/s eta 0:00:40
   - -------------------------------------- 1.3/42.9 MB 1.2 MB/s eta 0:00:36
   - -------------------------------------- 1.8/42.9 MB 1.4 MB/s eta 0:00:30
   - -------------------------------------- 2.1/42.9 MB 1.4 MB/s eta 0:00:30
   -- ------------------------------------- 2.4/42.9 MB 1.4 MB/s eta 0:00:29
   -- ------------------------------------- 2.6/42.9 MB 1.4 MB/s eta 0:00:30
   -- ------------------------------------- 2.6/42.9 MB 1.4 MB/s eta 0:00:30
   -- ------------------------------------- 2.9/42.9 MB 1.3 MB/s eta 0:00:32
   -- ----------

In [2]:
# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [3]:
# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")

In [4]:
# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")



Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully


In [5]:
# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [6]:
# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader("data_privacy_regulation.pdf")

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."

In [7]:
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs

In [8]:
# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result

In [9]:
# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer




In [10]:
# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )




In [ ]:
# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Processing PDF...
Total chunks: 7


Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Retrieved Chunks:
 ["es for Non■Compliance\nOrganizations found to be in violation of this regulation may face financial penalties, regulatory\ninvestigations, and operational restrictions. Administrative fines may reach up to 4 percent of the\ncompany's annual global revenue or a fixed penalty determined by the regulatory authority.\nRepeated violations may result in suspension of data processing activities.\nArticle 7: Compliance and Audits\nOrganizations must maintain documentation demonstrating compliance with this re", ' binding corporate rules, or regulatory approvals before\ntransferring personal data internationally.\nArticle 5: Data Security and Breach Notification\n\x7f\nOrganizations must implement encryption, access control, and monitoring systems.\n\x7f\nAny data breach that risks individual privacy must be reported to authorities within 72 hours.\n\x7f\nAffected individuals must be informed if the breach poses a high risk to their rights and\nfreedoms.\nArticle 6: Penalt

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Raw Model Output:
 
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr ing personal data.

Personal data should be collected for specified, explicit, and legitimate purposes.

Data minimization principles must be followed, ensuring only necessary data is collected.

Organizations must implement appropriate technical and organizational security measures.
Article 3: Rights of Data Subjects

Individuals have the right to access their personal d

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Chunks:
 ['Data Privacy and Protection Regulation (Sample\n Document)\nThis sample regulation is created for educational and demonstration purposes. It simulates a legal\ndocument that can be used for testing Retrieval-Augmented Generation (RAG) systems. The\nregulation outlines rules related to data collection, storage, processing, cross-border transfers, and\npenalties for non■compliance.\nArticle 1: Definitions\n\x7f\nPersonal Data: Any information relating to an identified or identifiable individual.\n\x7f\nData Contr', 'ing personal data.\n\x7f\nPersonal data should be collected for specified, explicit, and legitimate purposes.\n\x7f\nData minimization principles must be followed, ensuring only necessary data is collected.\n\x7f\nOrganizations must implement appropriate technical and organizational security measures.\nArticle 3: Rights of Data Subjects\n\x7f\nIndividuals have the right to access their personal data.\n\x7f\nIndividuals may request correction of inaccurat

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Chunks:
 ['Data Privacy and Protection Regulation (Sample\n Document)\nThis sample regulation is created for educational and demonstration purposes. It simulates a legal\ndocument that can be used for testing Retrieval-Augmented Generation (RAG) systems. The\nregulation outlines rules related to data collection, storage, processing, cross-border transfers, and\npenalties for non■compliance.\nArticle 1: Definitions\n\x7f\nPersonal Data: Any information relating to an identified or identifiable individual.\n\x7f\nData Contr', 'ing personal data.\n\x7f\nPersonal data should be collected for specified, explicit, and legitimate purposes.\n\x7f\nData minimization principles must be followed, ensuring only necessary data is collected.\n\x7f\nOrganizations must implement appropriate technical and organizational security measures.\nArticle 3: Rights of Data Subjects\n\x7f\nIndividuals have the right to access their personal data.\n\x7f\nIndividuals may request correction of inaccurat